# Ernesto Investing AI — Notebook 05: Analisis de Sentimiento NLP (VADER)

Analiza el sentimiento de titulares financieros con VADER (NLTK) y
clasifica cada uno como BULLISH, BEARISH o NEUTRAL. La especificacion
del curso autoriza explicitamente usar un dataset simulado pero
realista si no se dispone de una fuente de noticias real — por eso
este notebook no depende de ninguna API externa de noticias, lo que
lo hace robusto para ejecutarse en la demo sin riesgo de conexion rota.

**Requisito previo:** ninguno (es independiente de los demas notebooks).

## 1. Instalacion e importaciones

In [ ]:
!pip install "pymongo[srv]" nltk --quiet

In [ ]:
import random
from datetime import datetime, timedelta, timezone

import nltk
nltk.download("vader_lexicon", quiet=True)
from nltk.sentiment import SentimentIntensityAnalyzer

from pymongo import MongoClient
from google.colab import userdata

random.seed(42)
analizador = SentimentIntensityAnalyzer()

## 2. Configuracion y conexion a MongoDB

In [ ]:
TICKERS = ["FSM", "VOLCABC1.LM", "ABX.TO", "BVN", "BHP"]
NOMBRES_EMPRESA = {
    "FSM": "Fortuna Silver Mines",
    "VOLCABC1.LM": "Volcan Compania Minera",
    "ABX.TO": "Barrick Gold",
    "BVN": "Compania de Minas Buenaventura",
    "BHP": "BHP Billiton",
}
FUENTES = ["Reuters", "Bloomberg", "MarketWatch", "Yahoo Finance", "Investing.com"]
NOTICIAS_POR_TICKER = 15

MONGO_URI = userdata.get("MONGO_URI")
MONGO_DB_NAME = "ernesto_investing_ai"

cliente = MongoClient(MONGO_URI)
db = cliente[MONGO_DB_NAME]
col_sentimiento = db["sentimiento_noticias"]

print("Conexion exitosa a MongoDB Atlas.")

## 3. Dataset simulado de titulares financieros

Plantillas realistas en ingles (VADER esta entrenado sobre texto en
ingles) con tono positivo, negativo y neutral. Se combinan con el
nombre de cada empresa para generar titulares variados.

In [ ]:
PLANTILLAS_POSITIVAS = [
    "{empresa} shares surge after strong quarterly earnings report",
    "{empresa} beats analyst expectations, stock climbs sharply",
    "Analysts upgrade {empresa} citing robust production growth",
    "{empresa} announces record revenue for the quarter",
    "Investors optimistic as {empresa} expands mining operations",
]

PLANTILLAS_NEGATIVAS = [
    "{empresa} shares plunge after disappointing production numbers",
    "{empresa} misses earnings estimates, stock drops",
    "Analysts downgrade {empresa} amid rising operational costs",
    "{empresa} faces regulatory setback in key mining region",
    "Investors concerned as {empresa} reports declining margins",
]

PLANTILLAS_NEUTRALES = [
    "{empresa} reports quarterly results in line with expectations",
    "{empresa} schedules annual shareholder meeting for next month",
    "{empresa} announces changes to its board of directors",
    "{empresa} to present at upcoming mining industry conference",
    "{empresa} maintains guidance for the current fiscal year",
]


def generar_titular(empresa: str) -> str:
    """Elige aleatoriamente una plantilla positiva, negativa o neutral."""
    categoria = random.choice(["positiva", "negativa", "neutral"])
    plantilla = {
        "positiva": random.choice(PLANTILLAS_POSITIVAS),
        "negativa": random.choice(PLANTILLAS_NEGATIVAS),
        "neutral": random.choice(PLANTILLAS_NEUTRALES),
    }[categoria]
    return plantilla.format(empresa=empresa)

## 4. Analisis de sentimiento y clasificacion

Umbrales estandar de VADER: BULLISH si compound >= 0.05,
BEARISH si compound <= -0.05, NEUTRAL en el resto de los casos.

In [ ]:
def clasificar_compound(compound: float) -> str:
    if compound >= 0.05:
        return "BULLISH"
    elif compound <= -0.05:
        return "BEARISH"
    return "NEUTRAL"


def generar_noticias_ticker(ticker: str, empresa: str, cantidad: int = NOTICIAS_POR_TICKER) -> list[dict]:
    """Genera un lote de noticias simuladas ya analizadas con VADER para un ticker."""
    documentos = []
    for _ in range(cantidad):
        titular = generar_titular(empresa)
        compound = analizador.polarity_scores(titular)["compound"]
        fecha = datetime.now(timezone.utc) - timedelta(days=random.randint(0, 29))

        documentos.append({
            "ticker": ticker,
            "titular": titular,
            "fuente": random.choice(FUENTES),
            "fecha": fecha,
            "compound": round(compound, 4),
            "clasificacion": clasificar_compound(compound),
        })
    return documentos

## 5. Ejecucion: generar, analizar y guardar en MongoDB

Antes de insertar el lote nuevo, se eliminan las noticias simuladas
previas de cada ticker para que el notebook sea idempotente (se puede
re-ejecutar sin acumular duplicados en cada corrida).

In [ ]:
resumen = {}

for ticker in TICKERS:
    empresa = NOMBRES_EMPRESA[ticker]
    print(f"Procesando {ticker} ({empresa})...")

    col_sentimiento.delete_many({"ticker": ticker})
    documentos = generar_noticias_ticker(ticker, empresa)
    col_sentimiento.insert_many(documentos)

    promedio_compound = sum(d["compound"] for d in documentos) / len(documentos)
    conteo = {
        "BULLISH": sum(1 for d in documentos if d["clasificacion"] == "BULLISH"),
        "BEARISH": sum(1 for d in documentos if d["clasificacion"] == "BEARISH"),
        "NEUTRAL": sum(1 for d in documentos if d["clasificacion"] == "NEUTRAL"),
    }
    resumen[ticker] = {"promedio_compound": round(promedio_compound, 3), "conteo": conteo}
    print(f"  {ticker}: {len(documentos)} noticias, promedio compound={promedio_compound:.3f}, {conteo}")

print()
print("Resumen final:", resumen)

## 6. Verificacion final

In [ ]:
print("Total de noticias guardadas:", col_sentimiento.count_documents({}))
print()

for ticker in TICKERS:
    ejemplo = col_sentimiento.find_one({"ticker": ticker})
    if ejemplo:
        print(f"{ticker}: \"{ejemplo['titular']}\" -> {ejemplo['clasificacion']} ({ejemplo['compound']})")